# Evaluation: held-out months of the training year

In each split the Model A components are trained on 10 months and predict the 2 held-out months (Jan+Jul, Feb+Aug, Jun+Dec). The predictions for these 6 months therefore come from models that never saw them (`artifacts/oof/model_a_oof.parquet`). RMSE in taxi seconds:
- **NM baseline:** taxi = MVT − AOBT_3, i.e. the NM off-block estimate taken as correct.
- **components and blend:** taxi = max(60, (MVT − AOBT_3) − offset).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
import config
import run
print("raw data folder:", config.RAW.name)

raw data folder: data


In [2]:
from taxi.correctors import common as C
d = C.oof_base()
oof = pd.read_parquet(config.ARTIFACTS / "oof" / "model_a_oof.parquet")
d = d.merge(oof[["MVT_ID_mvt"] + list(config.MODEL_A)], on="MVT_ID_mvt")
y = d["y_taxi"].to_numpy(float)
rmse = lambda p, m=slice(None): float(np.sqrt(np.mean((np.asarray(p)[m] - y[m]) ** 2)))
result = {"NM baseline": rmse(d.NM_TAXI_DIFF_SEC)}
for name in config.MODEL_A:
    result[name] = rmse(np.maximum(60, d.NM_TAXI_DIFF_SEC - d[name]))
result["blend"] = rmse(d.base)
pd.Series(result, name="RMSE (s), 6 months").round(1)

NM baseline           398.5
xgboost               211.2
catboost_symmetric    209.4
lightgbm_lock         210.0
lightgbm_huber        209.4
catboost_depthwise    209.4
blend                 206.0
Name: RMSE (s), 6 months, dtype: float64

In [3]:
table = {}
for apt, g in d.groupby("ADEP_mvt").groups.items():
    m = d.index.isin(g)
    table[apt] = {"rows": int(m.sum()), "NM baseline": rmse(d.NM_TAXI_DIFF_SEC, m), "blend": rmse(d.base, m)}
pd.DataFrame(table).T.round(1)

,rows,NM baseline,blend
EDDF,110176.0,265.2,156.9
EDDM,80041.0,383.1,157.5
EGLL,118187.0,446.9,210.2
EHAM,119981.0,335.3,150.4
LEBL,86832.0,313.9,195.3
LEMD,104620.0,271.6,161.6
LFPG,116991.0,387.5,219.0
LIRF,78529.0,588.7,348.7
LSZH,65389.0,283.0,158.8
LTFM,134442.0,534.0,237.5


In [4]:
pd.DataFrame({m: {"rows": int((d.MVT_MONTH == m).sum()), "NM baseline": rmse(d.NM_TAXI_DIFF_SEC, (d.MVT_MONTH == m).to_numpy()),
                  "blend": rmse(d.base, (d.MVT_MONTH == m).to_numpy())} for m in sorted(d.MVT_MONTH.unique())}).T.round(1)

,rows,NM baseline,blend
1,152271.0,377.0,193.8
2,142622.0,367.5,207.9
6,180640.0,390.9,204.1
7,186775.0,465.8,230.3
8,188852.0,393.1,204.8
12,164028.0,374.6,188.5


## Model B (rows without an NM off-block time)
Out-of-fold predictions of Model B in the training year (`artifacts/oof/model_b_oof.parquet`): the per-airport method (EHAM / LSZH / EDDF / LIRF) and the all-rows model (other airports).

In [5]:
from taxi.features import tables as T
b = T.base("train")
b = b[b.NM_TAXI_DIFF_SEC.isna() & b.TAXITIME_SEC_mvt.notna()][["MVT_ID_mvt", "ADEP_mvt", "TAXITIME_SEC_mvt"]]
b = b.merge(pd.read_parquet(config.ARTIFACTS / "oof" / "model_b_oof.parquet"), on="MVT_ID_mvt", how="left")
b["base"] = np.where(b.ADEP_mvt.isin(["EHAM", "LSZH", "EDDF", "LIRF"]), b.base_pa, b.base_all)
b.groupby("ADEP_mvt").apply(lambda g: pd.Series({"rows": len(g), "RMSE": np.sqrt(np.mean((g.base - g.TAXITIME_SEC_mvt) ** 2)),
                                                 "median |error|": np.median(np.abs(g.base - g.TAXITIME_SEC_mvt))}),
                            include_groups=False).round(1)

,rows,RMSE,median |error|
ADEP_mvt,,,
EDDF,1956.0,315.2,143.5
EDDM,1010.0,520.8,166.5
EGLL,1414.0,899.0,231.1
EHAM,4084.0,353.1,106.9
LEBL,1862.0,300.8,148.0
LEMD,947.0,263.6,131.7
LFPG,3773.0,1735.7,195.0
LIRF,1488.0,4350.7,764.5
LSZH,2291.0,1913.2,136.7
